# Multi-Task Customer Support Transformer Architecture
### Production Training: 5 Epochs + Early Stopping + Best Model Checkpoint Saving
1. **Category Classification Head** (11 classes)
2. **Intent Classification Head** (27 classes)
3. **Linguistic & Emotion Flags Head** (14 multi-hot flags trained with `pos_weight`)
4. **NER / Slot Filling Head** (19 BIO classes for 9 entity slots with **Synthetic Value Injection**)
5. **Escalation / Urgency Neural Regressor** (Pure Neural Output of `self.escalation_head`)
6. **Customer Effort Score (CES) Neural Regressor** (Pure Neural Output of `self.effort_head`)
7. **Multi-Turn Conversation Trajectory Tracking**

In [1]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Suppress informational hub warnings
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})')

Using Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti (CUDA 12.4)


## 1. Load Dataset & Calculate Class Balancing Weights

In [2]:
CSV_PATH = r'../archive/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv'
df = pd.read_csv(CSV_PATH)
print(f'Dataset Shape: {df.shape}')

# 1. Categories & Intents Mapping
CATEGORIES = sorted(df['category'].unique().tolist())
INTENTS = sorted(df['intent'].unique().tolist())
cat2id = {c: i for i, c in enumerate(CATEGORIES)}
intent2id = {intent: i for i, intent in enumerate(INTENTS)}
id2cat = {i: c for c, i in cat2id.items()}
id2intent = {i: intent for intent, i in intent2id.items()}

df['target_category'] = df['category'].map(cat2id)
df['target_intent'] = df['intent'].map(intent2id)

# 2. Linguistic Flags Mapping & Class Weights (14 features)
FLAGS = ['B', 'L', 'Q', 'I', 'Z', 'M', 'C', 'K', 'E', 'P', 'W', 'N', 'S', 'V']
flag2id = {f: i for i, f in enumerate(FLAGS)}

FLAG_DEFINITIONS = {
    'W': 'Anger / Hostility / Swearing',
    'M': 'Frustration / Friction',
    'E': 'Emotional Distress / Vulnerability',
    'S': 'Sarcasm / Sharp Tone',
    'N': 'Negation / Refusal',
    'Z': 'Typo / Rushed Typing',
    'L': 'Politeness / Courtesy',
    'I': 'Indirect / Polite Inquiry',
    'Q': 'Direct Question',
    'C': 'Colloquial / Slang',
    'K': 'Telegraphic / Keyword (Neutral)',
    'P': 'Punctuation Defects',
    'B': 'Base Query Structure',
    'V': 'Vernacular Phrasing'
}

# Compute positive class weights for all 14 flags to balance training gradients
pos_weights = []
for f in FLAGS:
    df[f'flag_{f}'] = df['flags'].apply(lambda x: 1.0 if f in str(x) else 0.0)
    pos = df[f'flag_{f}'].sum()
    neg = len(df) - pos
    w = min(15.0, max(1.0, neg / max(pos, 1.0)))
    pos_weights.append(w)

FLAG_POS_WEIGHTS = torch.tensor(pos_weights, dtype=torch.float)
print(f'Computed Pos Weights for Flags: {dict(zip(FLAGS, [round(x, 1) for x in pos_weights]))}')

# 3. Target Escalation Score [0.0 to 1.0]
high_risk_intents = {'complaint', 'payment_issue', 'registration_problems', 'delete_account'}
df['is_high_risk'] = df['intent'].apply(lambda x: 1.0 if x in high_risk_intents else 0.0)

df['target_escalation_score'] = (
    0.45 * df['flag_W'] +
    0.35 * df['flag_M'] +
    0.25 * df['flag_E'] +
    0.20 * df['is_high_risk'] +
    0.10 * df['flag_N'] -
    0.15 * (df['flag_L'] * (1 - df['flag_M']))
).clip(0.0, 1.0).round(3)

# 4. Target Customer Effort Score (CES) [0.0 to 1.0]
df['text_len_norm'] = (df['instruction'].str.len() / 150.0).clip(0.0, 1.0)
df['target_customer_effort'] = (
    0.30 * df['flag_Z'] +
    0.25 * df['flag_E'] +
    0.15 * df['flag_P'] +
    0.20 * df['flag_M'] +
    0.15 * df['text_len_norm'] -
    0.20 * df['flag_K']
).clip(0.0, 1.0).round(3)

# 5. Target Message Completeness
df['has_entity'] = df['instruction'].apply(lambda x: 1 if re.search(r'\{\{([^}]+)\}\}', str(x)) else 0)
df['target_completeness'] = df.apply(lambda r: 1.0 if (r['has_entity'] == 1 or r['flag_K'] == 0) else 0.0, axis=1)

# 6. NER Slots & BIO Labels (9 entities -> 19 BIO classes)
raw_entities = set()
for inst in df['instruction']:
    for match in re.findall(r'\{\{([^}]+)\}\}', str(inst)):
        raw_entities.add(match)
ENTITIES = sorted(list(raw_entities))

ner_labels = ['O']
for ent in ENTITIES:
    ent_tag = ent.replace(' ', '_')
    ner_labels.extend([f'B-{ent_tag}', f'I-{ent_tag}'])

ner2id = {l: i for i, l in enumerate(ner_labels)}
id2ner = {i: l for l, i in ner2id.items()}

print(f'Categories ({len(CATEGORIES)}): {CATEGORIES[:5]}...')
print(f'Intents ({len(INTENTS)}): {INTENTS[:5]}...')
print(f'NER Classes ({len(ner2id)}): {list(ner2id.keys())[:7]}...')

Dataset Shape: (26872, 5)
Computed Pos Weights for Flags: {'B': 1.0, 'L': 1.0, 'Q': np.float64(2.0), 'I': np.float64(2.4), 'Z': np.float64(4.1), 'M': np.float64(4.5), 'C': np.float64(9.2), 'K': np.float64(11.1), 'E': np.float64(13.3), 'P': 15.0, 'W': 15.0, 'N': 15.0, 'S': 15.0, 'V': 15.0}
Categories (11): ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK']...
Intents (27): ['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice']...
NER Classes (19): ['O', 'B-Account_Category', 'I-Account_Category', 'B-Account_Type', 'I-Account_Type', 'B-Currency_Symbol', 'I-Currency_Symbol']...


## 2. Synthetic Slot Injection & Multi-Task Dataset
> **Synthetic Injection**: Dynamically injects randomized real-world values (`ORD-98124`, `$150.00`, `Chicago`, `INV-88214`) while precisely tracking subword token boundaries.

In [3]:
# Synthetic Entity Value Generators
SYNTHETIC_ENTITIES = {
    'Order Number': lambda: random.choice([
        f'ORD-{random.randint(10000, 99999)}',
        f'#{random.randint(100000, 999999)}',
        f'{random.randint(1000000, 9999999)}',
        f'PO-{random.randint(1000, 9999)}'
    ]),
    'Refund Amount': lambda: random.choice([
        f'${random.randint(10, 500)}.{random.randint(10, 99)}',
        f'${random.randint(15, 250)}',
        f'{random.randint(20, 1000)} USD',
        f'€{random.randint(10, 300)}'
    ]),
    'Invoice Number': lambda: f'INV-{random.randint(10000, 99999)}',
    'Delivery City': lambda: random.choice(['New York', 'Chicago', 'London', 'Berlin', 'San Francisco', 'Austin', 'Seattle']),
    'Delivery Country': lambda: random.choice(['USA', 'Canada', 'Germany', 'United Kingdom', 'France']),
    'Person Name': lambda: random.choice(['John Doe', 'Sarah Connor', 'Alex Smith', 'Emma Watson', 'David Miller']),
    'Account Category': lambda: random.choice(['Personal', 'Business', 'Enterprise', 'VIP']),
    'Account Type': lambda: random.choice(['Standard', 'Premium', 'Pro', 'Basic']),
    'Currency Symbol': lambda: random.choice(['$', '€', '£', 'USD', 'EUR'])
}

def inject_synthetic_entities(text: str):
    if 'invoice' in text.lower() and '{{Order Number}}' in text:
        text = text.replace('{{Order Number}}', '{{Invoice Number}}')
        
    pattern = re.compile(r'\{\{([^}]+)\}\}')
    spans = []
    new_text = ''
    last_end = 0
    
    for match in pattern.finditer(text):
        ent_name = match.group(1).strip()
        generator = SYNTHETIC_ENTITIES.get(ent_name, lambda: ent_name)
        val = generator()
        
        new_text += text[last_end:match.start()]
        start_char = len(new_text)
        new_text += val
        end_char = len(new_text)
        
        spans.append((start_char, end_char, ent_name.replace(' ', '_')))
        last_end = match.end()
        
    new_text += text[last_end:]
    return new_text, spans

MODEL_NAME = 'distilbert-base-uncased'  # Fast and lightweight backbone
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CustomerSupportMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64, augment_slots=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment_slots = augment_slots
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_text = str(row['instruction'])
        
        if self.augment_slots:
            text, entity_spans = inject_synthetic_entities(raw_text)
        else:
            text = raw_text
            entity_spans = [
                (m.start(), m.end(), m.group(1).replace(' ', '_'))
                for m in re.finditer(r'\{\{([^}]+)\}\}', text)
            ]
            
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_offsets_mapping=True,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        offsets = encoding['offset_mapping'].squeeze(0)
        
        ner_tags = torch.full((self.max_len,), fill_value=-100, dtype=torch.long)
        
        for token_idx, (start, end) in enumerate(offsets):
            if start == end:
                continue
            assigned_tag = ner2id['O']
            for span_start, span_end, ent_name in entity_spans:
                if start >= span_start and end <= span_end:
                    if start == span_start or token_idx == 0 or ner_tags[token_idx - 1] == -100 or ner_tags[token_idx - 1] == ner2id['O']:
                        assigned_tag = ner2id.get(f'B-{ent_name}', ner2id['O'])
                    else:
                        assigned_tag = ner2id.get(f'I-{ent_name}', ner2id['O'])
                    break
            ner_tags[token_idx] = assigned_tag
            
        flags_vec = torch.tensor([row[f'flag_{f}'] for f in FLAGS], dtype=torch.float)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'category_label': torch.tensor(row['target_category'], dtype=torch.long),
            'intent_label': torch.tensor(row['target_intent'], dtype=torch.long),
            'flags_label': flags_vec,
            'escalation_label': torch.tensor(row['target_escalation_score'], dtype=torch.float),
            'effort_label': torch.tensor(row['target_customer_effort'], dtype=torch.float),
            'completeness_label': torch.tensor(row['target_completeness'], dtype=torch.float),
            'ner_labels': ner_tags
        }

## 3. Multi-Task Transformer Model Architecture

In [4]:
class MultiTaskCustomerSupportNLU(nn.Module):
    def __init__(
        self,
        model_name=MODEL_NAME,
        num_categories=len(CATEGORIES),
        num_intents=len(INTENTS),
        num_flags=len(FLAGS),
        num_ner_labels=len(ner2id),
        pos_weights=FLAG_POS_WEIGHTS,
        dropout_rate=0.2
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        
        # HEAD 1: Category Classifier (11 classes)
        self.category_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_categories)
        )
        
        # HEAD 2: Intent Classifier (27 classes)
        self.intent_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_intents)
        )
        
        # HEAD 3: Sentiment & Nuance Multi-label Flags (14 binary features with class-weights)
        self.flags_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_flags)
        )
        
        # HEAD 4: Escalation / Urgency Neural Regressor [0.0, 1.0]
        self.escalation_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # HEAD 5: Customer Effort Score (CES) Neural Regressor [0.0, 1.0]
        self.effort_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # HEAD 6: Message Completeness (Binary Classifier)
        self.completeness_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.GELU(),
            nn.Linear(32, 1)
        )
        
        # HEAD 7: Token-Level NER / Slot Filling (19 BIO tags)
        self.ner_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size // 2, num_ner_labels)
        )
        
        # Loss functions
        self.ce_loss = nn.CrossEntropyLoss()
        # Flags head gets the (14,) pos_weight tensor
        self.flags_bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(DEVICE))
        # Completeness head gets standard scalar BCE loss
        self.comp_bce_loss = nn.BCEWithLogitsLoss()
        self.smooth_l1 = nn.SmoothL1Loss()
        self.ner_loss = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(
        self,
        input_ids,
        attention_mask,
        category_label=None,
        intent_label=None,
        flags_label=None,
        escalation_label=None,
        effort_label=None,
        completeness_label=None,
        ner_labels=None
    ):
        encoder_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq_output = encoder_out.last_hidden_state
        cls_output = seq_output[:, 0, :]
        
        cls_dropped = self.dropout(cls_output)
        seq_dropped = self.dropout(seq_output)
        
        cat_logits = self.category_head(cls_dropped)
        intent_logits = self.intent_head(cls_dropped)
        flags_logits = self.flags_head(cls_dropped)
        escalation_pred = self.escalation_head(cls_output).squeeze(-1)
        effort_pred = self.effort_head(cls_output).squeeze(-1)
        completeness_logits = self.completeness_head(cls_dropped).squeeze(-1)
        ner_logits = self.ner_head(seq_dropped)
        
        total_loss = None
        losses = {}
        
        if category_label is not None:
            l_cat = self.ce_loss(cat_logits, category_label)
            l_intent = self.ce_loss(intent_logits, intent_label)
            l_flags = self.flags_bce_loss(flags_logits, flags_label)
            l_esc = self.smooth_l1(escalation_pred, escalation_label)
            l_eff = self.smooth_l1(effort_pred, effort_label)
            l_comp = self.comp_bce_loss(completeness_logits, completeness_label)
            l_ner = self.ner_loss(ner_logits.view(-1, ner_logits.shape[-1]), ner_labels.view(-1))
            
            total_loss = (
                1.0 * l_intent +
                0.5 * l_cat +
                1.0 * l_flags +
                1.2 * l_esc +
                1.0 * l_eff +
                0.5 * l_comp +
                1.5 * l_ner
            )
            
            losses = {
                'total': total_loss.item(),
                'intent': l_intent.item(),
                'category': l_cat.item(),
                'flags': l_flags.item(),
                'escalation': l_esc.item(),
                'effort': l_eff.item(),
                'ner': l_ner.item()
            }
            
        return {
            'loss': total_loss,
            'losses': losses,
            'cat_logits': cat_logits,
            'intent_logits': intent_logits,
            'flags_logits': flags_logits,
            'escalation_pred': escalation_pred,
            'effort_pred': effort_pred,
            'completeness_logits': completeness_logits,
            'ner_logits': ner_logits
        }

## 4. Train / Validation Split & DataLoaders

In [5]:
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['target_intent'])
print(f'Train samples: {len(train_df)} | Val samples: {len(val_df)}')

train_dataset = CustomerSupportMultiTaskDataset(train_df, tokenizer, max_len=64, augment_slots=True)
val_dataset = CustomerSupportMultiTaskDataset(val_df, tokenizer, max_len=64, augment_slots=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = MultiTaskCustomerSupportNLU().to(DEVICE)
print(f'Model initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.')

Train samples: 22841 | Val samples: 4031


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Model initialized with 67,694,602 trainable parameters.


## 5. Training Engine with Early Stopping & Best Model Checkpointing
> Trains for up to **5 Epochs** with **Early Stopping (Patience = 2)** and automatically saves the best performing model weights to `best_multitask_nlu.pt`.

In [6]:
EPOCHS = 5
PATIENCE = 2
BEST_MODEL_PATH = 'best_multitask_nlu.pt'

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    
    for batch in pbar:
        optimizer.zero_grad()
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        out = model(**batch)
        loss = out['loss']
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Evaluation Phase
    model.eval()
    total_val_loss = 0.0
    correct_cat = 0
    correct_intent = 0
    total_samples = 0
    esc_errors = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            
            total_val_loss += out['loss'].item()
            cat_preds = out['cat_logits'].argmax(dim=-1)
            intent_preds = out['intent_logits'].argmax(dim=-1)
            
            correct_cat += (cat_preds == batch['category_label']).sum().item()
            correct_intent += (intent_preds == batch['intent_label']).sum().item()
            total_samples += batch['category_label'].size(0)
            
            esc_errors.extend(torch.abs(out['escalation_pred'] - batch['escalation_label']).cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_cat_acc = correct_cat / total_samples
    val_intent_acc = correct_intent / total_samples
    val_esc_mae = np.mean(esc_errors)
    
    print(f'\n--- Epoch {epoch+1} Results ---')
    print(f'Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    print(f'Val Category Accuracy: {val_cat_acc * 100:.2f}%')
    print(f'Val Intent Accuracy:   {val_intent_acc * 100:.2f}%')
    print(f'Val Escalation MAE:    {val_esc_mae:.4f}')
    
    # Early Stopping & Best Checkpoint Logic
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'🌟 [CHECKPOINT] New best model saved to {BEST_MODEL_PATH} (Val Loss: {best_val_loss:.4f})')
    else:
        patience_counter += 1
        print(f'⚠️ [PATIENCE] No loss improvement for {patience_counter}/{PATIENCE} epoch(s).')
        if patience_counter >= PATIENCE:
            print(f'🛑 [EARLY STOPPING] Triggered at Epoch {epoch+1}! Stopping training early.')
            break
    print('-' * 45)

# Load the best model weights for inference
print(f'\nRestoring best model checkpoint from {BEST_MODEL_PATH}...')
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()
print('✅ Model ready with optimal weights for real-time inference!')

Epoch 1/5 [Train]:   0%|          | 0/714 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]


--- Epoch 1 Results ---
Train Loss: 3.2937 | Val Loss: 0.4975
Val Category Accuracy: 99.88%
Val Intent Accuracy:   99.38%
Val Escalation MAE:    0.0643
🌟 [CHECKPOINT] New best model saved to best_multitask_nlu.pt (Val Loss: 0.4975)
---------------------------------------------


Epoch 2/5 [Train]:   0%|          | 0/714 [00:00<?, ?it/s]

Epoch 2/5 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]

...

Epoch 4/5 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]


--- Epoch 4 Results ---
Train Loss: 0.1132 | Val Loss: 0.1104
Val Category Accuracy: 99.95%
Val Intent Accuracy:   99.75%
Val Escalation MAE:    0.0232
🌟 [CHECKPOINT] New best model saved to best_multitask_nlu.pt (Val Loss: 0.1104)
---------------------------------------------


Epoch 5/5 [Train]:   0%|          | 0/714 [00:00<?, ?it/s]

Epoch 5/5 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]


--- Epoch 5 Results ---
Train Loss: 0.0965 | Val Loss: 0.1056
Val Category Accuracy: 99.95%
Val Intent Accuracy:   99.75%
Val Escalation MAE:    0.0223
🌟 [CHECKPOINT] New best model saved to best_multitask_nlu.pt (Val Loss: 0.1056)
---------------------------------------------

Restoring best model checkpoint from best_multitask_nlu.pt...
✅ Model ready with optimal weights for real-time inference!


## 6. Real-Time Inference Engine (100% Pure Neural Network Outputs)

In [7]:
def predict_nlu_detailed(text, history=None):
    model.eval()
    
    # 1. Encode current message for emotion & slot extraction
    current_encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=64,
        return_offsets_mapping=True,
        return_tensors='pt'
    )
    
    # 2. Encode context for intent & category disambiguation
    full_context = text
    if history:
        history_str = ' '.join([f"{h['role']}: {h['text']}" for h in history[-3:]])
        full_context = f"{history_str} Current Customer: {text}"
        
    context_encoding = tokenizer(
        full_context,
        padding='max_length',
        truncation=True,
        max_length=64,
        return_offsets_mapping=True,
        return_tensors='pt'
    )
    
    with torch.no_grad():
        out_current = model(
            input_ids=current_encoding['input_ids'].to(DEVICE),
            attention_mask=current_encoding['attention_mask'].to(DEVICE)
        )
        out_context = model(
            input_ids=context_encoding['input_ids'].to(DEVICE),
            attention_mask=context_encoding['attention_mask'].to(DEVICE)
        )
        
    # 100% PURE NEURAL HEAD PREDICTIONS:
    cat_idx = out_context['cat_logits'].argmax(dim=-1).item()
    intent_idx = out_context['intent_logits'].argmax(dim=-1).item()
    
    # Escalation & CES directly from the Transformer's Regression Heads (Head 4 & Head 5)
    neural_escalation = out_current['escalation_pred'].item()
    neural_effort = out_current['effort_pred'].item()
    neural_completeness = torch.sigmoid(out_current['completeness_logits']).item() > 0.5
    
    # Emotion Probabilities from Head 3
    flag_probs = torch.sigmoid(out_current['flags_logits']).squeeze(0).cpu().numpy()
    
    raw_emotions = {
        'Anger / Hostility (W)': float(flag_probs[flag2id['W']]),
        'Frustration / Friction (M)': float(flag_probs[flag2id['M']]),
        'Emotional Distress (E)': float(flag_probs[flag2id['E']]),
        'Sarcasm / Sharp Tone (S)': float(flag_probs[flag2id['S']]),
        'Polite Courtesy (I)': float(flag_probs[flag2id['I']]),
        'Neutral / Direct': max(0.02, 1.0 - (float(flag_probs[flag2id['W']]) + float(flag_probs[flag2id['M']]) + float(flag_probs[flag2id['E']] + float(flag_probs[flag2id['I']] / 2.0))))
    }
    
    total_score = sum(raw_emotions.values()) + 1e-6
    normalized_emotions = {k: f"{round((v / total_score) * 100, 1)}%" for k, v in raw_emotions.items()}
    
    # Extract NER Entities from Head 7
    ner_preds = out_current['ner_logits'].argmax(dim=-1).squeeze(0).cpu().numpy()
    offsets = current_encoding['offset_mapping'].squeeze(0).cpu().numpy()
    
    extracted_entities = {}
    current_entity = None
    current_start = None
    current_end = None
    
    for pred_id, (start, end) in zip(ner_preds, offsets):
        if start == end: continue
        tag = id2ner[pred_id]
        if tag.startswith('B-'):
            if current_entity and current_start is not None:
                extracted_entities[current_entity] = text[current_start:current_end].strip()
            current_entity = tag[2:]
            current_start = start
            current_end = end
        elif tag.startswith('I-') and current_entity == tag[2:]:
            current_end = end
        else:
            if current_entity and current_start is not None:
                extracted_entities[current_entity] = text[current_start:current_end].strip()
                current_entity = None
                current_start = None
                current_end = None
                
    if current_entity and current_start is not None:
        extracted_entities[current_entity] = text[current_start:current_end].strip()
        
    return {
        'Input Query': text,
        'Category': id2cat[cat_idx],
        'Intent': id2intent[intent_idx],
        'Neural Escalation Score': round(neural_escalation, 3),
        'Neural Customer Effort (CES)': round(neural_effort, 3),
        'Actionable / Complete': neural_completeness,
        'Normalized Emotion Profile (Sums to 100%)': normalized_emotions,
        'Raw Head Activations': {
            'Anger (W)': f"{round(float(flag_probs[flag2id['W']]) * 100, 1)}%",
            'Frustration (M)': f"{round(float(flag_probs[flag2id['M']]) * 100, 1)}%",
            'Distress (E)': f"{round(float(flag_probs[flag2id['E']]) * 100, 1)}%",
            'Courtesy (I)': f"{round(float(flag_probs[flag2id['I']]) * 100, 1)}%"
        },
        'Extracted Entities': extracted_entities
    }

## 7. Extended Single-Turn Test Suite

In [10]:
test_scenarios = [
    'This is completely unacceptable, cancel my damn subscription right now!',
    'I lost my job and I cannot afford purchase #99124, please help me cancel it',
    'I was charged twice $89.99 for order ORD-55421, I need my money back!',
    'Good morning, could you kindly let me know if you ship to Berlin?',
    'track order ORD-44912',
    'basically the issue is...',
    'I forgot my password for user Sarah Connor, how do I reset it?',
    'How can I upgrade my account from Basic to Premium?',
    'I am requesting a refund of $250.00 for my delivery to London'
]

print('=' * 80)
print('EXTENDED SINGLE-TURN TEST SUITE')
print('=' * 80)

for query in test_scenarios:
    res = predict_nlu_detailed(query)
    esc_alert = '🚨 [CRITICAL ALERT]' if res['Neural Escalation Score'] > 0.25 else '✅ [NORMAL]'
    
    print(f"\n🔹 QUERY : \"{res['Input Query']}\"")
    print(f"   ├─ Category       : {res['Category']}")
    print(f"   ├─ Intent         : {res['Intent']}")
    print(f"   ├─ Escalation     : {res['Neural Escalation Score']} {esc_alert}")
    print(f"   ├─ Customer Effort: {res['Neural Customer Effort (CES)']}")
    print(f"   ├─ Actionable     : {'Yes' if res['Actionable / Complete'] else '❌ No (Needs Clarification)'}")
    print(f"   ├─ Emotion Profile: {res['Normalized Emotion Profile (Sums to 100%)']}")
    print(f"   └─ Entities       : {res['Extracted Entities']}")
    print('-' * 80)

EXTENDED SINGLE-TURN TEST SUITE

🔹 QUERY : "This is completely unacceptable, cancel my damn subscription right now!"
   ├─ Category       : SUBSCRIPTION
   ├─ Intent         : newsletter_subscription
   ├─ Escalation     : 0.557 🚨 [CRITICAL ALERT]
   ├─ Customer Effort: 0.128
   ├─ Actionable     : Yes
   ├─ Emotion Profile: {'Anger / Hostility (W)': '75.8%', 'Frustration / Friction (M)': '0.9%', 'Emotional Distress (E)': '8.5%', 'Sarcasm / Sharp Tone (S)': '0.1%', 'Polite Courtesy (I)': '13.2%', 'Neutral / Direct': '1.5%'}
   └─ Entities       : {}
--------------------------------------------------------------------------------

🔹 QUERY : "I lost my job and I cannot afford purchase #99124, please help me cancel it"
   ├─ Category       : ORDER
   ├─ Intent         : cancel_order
   ├─ Escalation     : 0.151 ✅ [NORMAL]
   ├─ Customer Effort: 0.215
   ├─ Actionable     : Yes
   ├─ Emotion Profile: {'Anger / Hostility (W)': '1.8%', 'Frustration / Friction (M)': '2.0%', 'Emotional Distres

## 8. Multi-Turn Conversation Trajectory Simulation

In [11]:
chat_history = []
turns = [
    {'role': 'Customer', 'text': 'Hi, I have a question about cancelling order ORD-77881'},
    {'role': 'Agent',    'text': 'Could you please confirm your account email address?'},
    {'role': 'Customer', 'text': 'I already confirmed it twice! Why is this taking so long, cancel it damn it!'}
]

print('=' * 80)
print('MULTI-TURN TRAJECTORY SIMULATION')
print('=' * 80)

for i, turn in enumerate(turns):
    if turn['role'] == 'Customer':
        res = predict_nlu_detailed(turn['text'], history=chat_history)
        esc_alert = '🚨 [CRITICAL ALERT]' if res['Neural Escalation Score'] > 0.25 else '✅ [NORMAL]'
        
        print(f"\n[Turn {i+1}] Customer: \"{turn['text']}\"")
        print(f"  ├─ Category          : {res['Category']}")
        print(f"  ├─ Intent            : {res['Intent']}")
        print(f"  ├─ Dynamic Escalation: {res['Neural Escalation Score']} {esc_alert}")
        print(f"  ├─ Customer Effort   : {res['Neural Customer Effort (CES)']}")
        print(f"  ├─ Normalized Emotions (100%):")
        for emo, prob in res['Normalized Emotion Profile (Sums to 100%)'].items():
            print(f"  │    • {emo:28}: {prob}")
        print(f"  └─ Extracted Slots   : {res['Extracted Entities']}")
    chat_history.append(turn)

MULTI-TURN TRAJECTORY SIMULATION

[Turn 1] Customer: "Hi, I have a question about cancelling order ORD-77881"
  ├─ Category          : ORDER
  ├─ Intent            : cancel_order
  ├─ Dynamic Escalation: 0.021 ✅ [NORMAL]
  ├─ Customer Effort   : 0.081
  ├─ Normalized Emotions (100%):
  │    • Anger / Hostility (W)       : 0.9%
  │    • Frustration / Friction (M)  : 0.5%
  │    • Emotional Distress (E)      : 0.1%
  │    • Sarcasm / Sharp Tone (S)    : 2.9%
  │    • Polite Courtesy (I)         : 13.0%
  │    • Neutral / Direct            : 82.7%
  └─ Extracted Slots   : {'Order_Number': 'ORD-77881'}

[Turn 3] Customer: "I already confirmed it twice! Why is this taking so long, cancel it damn it!"
  ├─ Category          : ORDER
  ├─ Intent            : cancel_order
  ├─ Dynamic Escalation: 0.595 🚨 [CRITICAL ALERT]
  ├─ Customer Effort   : 0.274
  ├─ Normalized Emotions (100%):
  │    • Anger / Hostility (W)       : 43.7%
  │    • Frustration / Friction (M)  : 2.8%
  │    • Emotional Dist